In [ ]:
# Step 1: Install dependencies & set Hugging Face token
%pip install -q "datasets>=2.19.0" "huggingface_hub>=0.24"
import os
import getpass

# 直接设置Hugging Face token，跳过登录界面
hf_token = getpass.getpass("Paste your Hugging Face token: ")
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token

print("Hugging Face token set successfully!")

In [ ]:
# Step 2: Load FLARE-FSRL test set and preprocess
from datasets import load_dataset, Dataset

# FLARE-FSRL dataset (Financial Semantic Role Labeling)
print("加载 FLARE-FSRL 数据集...")
ds_raw = load_dataset("ChanceFocus/flare-fsrl", split="test")
print(f"成功加载! 样本数: {len(ds_raw)}, 列: {ds_raw.column_names}")

# 定义标签类型（从数据集中提取）
# FSRL 标签包括: O, B-QUANT, I-QUANT, B-TIME, I-TIME, B-MANNER, I-MANNER, 
# B-THEME, I-THEME, B-VALUE, I-VALUE, B-WHOLE, I-WHOLE, B-LOCATION, I-LOCATION,
# B-AGENT, I-AGENT, B-CAUSE, I-CAUSE, B-SOURCE, I-SOURCE, B-REF_TIME, I-REF_TIME,
# B-CONDITION, I-CONDITION

LABELS = ["O", "B-QUANT", "I-QUANT", "B-TIME", "I-TIME", "B-MANNER", "I-MANNER",
          "B-THEME", "I-THEME", "B-VALUE", "I-VALUE", "B-WHOLE", "I-WHOLE",
          "B-LOCATION", "I-LOCATION", "B-AGENT", "I-AGENT", "B-CAUSE", "I-CAUSE",
          "B-SOURCE", "I-SOURCE", "B-REF_TIME", "I-REF_TIME", "B-CONDITION", "I-CONDITION"]

def _map_row(x):
    tokens = x.get("token", [])
    labels = x.get("label", [])
    text = x.get("text", "")
    
    # 如果没有预分的 tokens，从 text 分词
    if not tokens and text:
        tokens = text.split()
    
    # 确保 tokens 和 labels 长度匹配
    if len(tokens) != len(labels):
        min_len = min(len(tokens), len(labels))
        tokens = tokens[:min_len]
        labels = labels[:min_len]
    
    return {
        "tokens": tokens,
        "labels": labels,
        "text": " ".join(tokens) if tokens else text,
        "label_types": LABELS
    }

ds = Dataset.from_list([_map_row(r) for r in ds_raw])
bad = [i for i, r in enumerate(ds) if not r["tokens"]]
print(f"空样本数: {len(bad)}")
assert len(bad) == 0, "发现空样本。"

print(f"\n第一个样本示例:")
print(f"  Tokens: {ds[0]['tokens'][:15]}...")
print(f"  Labels: {ds[0]['labels'][:15]}...")

In [ ]:
# Step 3: Install dependencies, configure OpenAI, and record experiment metadata
%pip install -q "openai==1.40.2" "httpx==0.27.2" "httpcore==1.0.5" \
               "pandas>=2.2.2" "tqdm>=4.66.4" "requests>=2.31.0"

import os, getpass, json, time, platform, re
from importlib.metadata import version, PackageNotFoundError
import requests

# o3-mini配置
MODEL = "o3-mini"
BASE_URL = "https://api.openai.com/v1"

api_key = os.getenv("OPENAI_API_KEY") or os.getenv("API_KEY")
if not api_key:
    api_key = getpass.getpass("Paste your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# 文件命名
dataset_name = "flare_fsrl"
run_tag = f"{dataset_name}_{MODEL.replace('-', '_')}"
save_dir = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
meta_path = f"{save_dir}/{run_tag}_metadata.json"

def ver(pkg: str) -> str:
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not-installed"

meta = {
    "dataset": "ChanceFocus/flare-fsrl",
    "split": "test",
    "label_types": LABELS,
    "model": MODEL,
    "openai_sdk": ver("openai"),
    "httpx": ver("httpx"),
    "httpcore": ver("httpcore"),
    "datasets_version": ver("datasets"),
    "pandas": ver("pandas"),
    "tqdm": ver("tqdm"),
    "time_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python": platform.python_version(),
    "base_url": BASE_URL,
    "note": "o3-mini evaluation on FLARE-FSRL (Financial Semantic Role Labeling)"
}

os.makedirs(save_dir, exist_ok=True)
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Meta saved ->", meta_path)
print("MODEL:", MODEL, "| BASE_URL:", BASE_URL)
print("OPENAI_API_KEY is set:", bool(os.environ.get("OPENAI_API_KEY")))

In [ ]:
# Step 4: Inference & evaluation loop for FSRL
import json, os, re, time
import pandas as pd
from tqdm import tqdm
import requests

def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _make_fsrl_prompt(tokens):
    text = " ".join(tokens)
    return (
        "Task: Perform Financial Semantic Role Labeling (FSRL) on the following text.\n\n"
        f"Text: {text}\n\n"
        "Instructions:\n"
        "1. For each token, identify its semantic role in the sentence\n"
        "2. Available roles: QUANT (quantity), TIME (time), MANNER (manner), THEME (theme), "
        "VALUE (value), WHOLE (whole), LOCATION (location), AGENT (agent), CAUSE (cause), "
        "SOURCE (source), REF_TIME (reference time), CONDITION (condition)\n"
        "3. Use BIO tagging: B-ROLE for beginning, I-ROLE for continuation\n"
        "4. Use 'O' for tokens without semantic roles\n"
        "5. Return the labels as a JSON array in the same order as tokens\n"
        "6. Return ONLY the JSON array, no additional text\n\n"
        "Example output format:\n"
        '["O", "B-TIME", "I-TIME", "O", "B-QUANT", "O", "B-VALUE", "I-VALUE"]'
    )

def parse_fsrl_response(response_text: str, num_tokens: int):
    response_text = response_text.strip()
    match = re.search(r'\[.*\]', response_text, re.DOTALL)
    if match:
        try:
            labels = json.loads(match.group())
            if isinstance(labels, list):
                if len(labels) > num_tokens:
                    labels = labels[:num_tokens]
                elif len(labels) < num_tokens:
                    labels = labels + ["O"] * (num_tokens - len(labels))
                return labels, True, None
        except json.JSONDecodeError:
            pass
    return None, False, "Failed to parse JSON array"

def ask_o3_mini_once(tokens):
    url = f"{BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    user_text = _make_fsrl_prompt(tokens)
    
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a semantic role labeling expert. Respond only with valid JSON arrays."},
            {"role": "user", "content": user_text}
        ]
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=60)
        
        if response.status_code != 200:
            return None, False, f"API Error {response.status_code}"
            
        result = response.json()
        
        if 'choices' not in result or len(result['choices']) == 0:
            return None, False, "No choices in response"
            
        txt = result['choices'][0]['message']['content']
        txt = _strip_code_fences(txt)
        pred_labels, success, error = parse_fsrl_response(txt, len(tokens))
        return pred_labels, success, error
        
    except Exception as e:
        return None, False, str(e)

def ask_o3_mini(tokens):
    delay = 2.0
    for attempt in range(6):
        try:
            pred_labels, success, error = ask_o3_mini_once(tokens)
            if success:
                return pred_labels, True, None
            else:
                if attempt < 5:
                    time.sleep(delay)
                    delay = min(delay * 2, 60)
                    continue
                return ["O"] * len(tokens), False, error
        except Exception as e:
            if attempt == 5:
                return ["O"] * len(tokens), False, str(e)
            time.sleep(delay)
            delay = min(delay * 2, 60)
    return ["O"] * len(tokens), False, "Exhausted retries"

run_tag = f"flare_fsrl_{MODEL.replace('-', '_')}"
save_dir = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
err_path = f"{save_dir}/{run_tag}_errors.csv"

rows_done = []
done_idx = set()
if os.path.exists(pred_path):
    old = pd.read_csv(pred_path)
    if "row_idx" in old.columns:
        rows_done = old.to_dict("records")
        done_idx = set(old["row_idx"].tolist())
        print(f"[resume] loaded {len(done_idx)} completed rows.")

err_rows = []
buf = []
save_every = 20

total = len(ds)
print(f"Starting o3-mini FSRL evaluation on {total} samples...")

for i in tqdm(range(total)):
    if i in done_idx:
        continue
    x = ds[i]
    tokens = x["tokens"]
    gold_labels = x["labels"]

    try:
        pred_labels, success, error_msg = ask_o3_mini(tokens)
        if not success:
            raise RuntimeError(error_msg)
        raw = json.dumps(pred_labels)
    except Exception as e:
        pred_labels = ["O"] * len(tokens)
        raw = f"ERROR: {type(e).__name__}: {e}"
        err_rows.append({"row_idx": i, "error": raw, "text": " ".join(tokens[:50])})
        success = False

    buf.append({
        "row_idx": i,
        "text": " ".join(tokens[:50]) + ("..." if len(tokens) > 50 else ""),
        "token_count": len(tokens),
        "pred_raw": raw,
        "pred_labels": json.dumps(pred_labels),
        "gold_labels": json.dumps(gold_labels),
        "success": success
    })

    if len(buf) % save_every == 0:
        out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
        out.to_csv(pred_path, index=False)
        if err_rows:
            pd.DataFrame(err_rows).to_csv(err_path, index=False)
        print(f"[checkpoint] saved {len(out)}/{total} -> {pred_path}")

out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
out.to_csv(pred_path, index=False)
if err_rows:
    pd.DataFrame(err_rows).to_csv(err_path, index=False)
print(f"[done] o3-mini FSRL evaluation completed -> {pred_path}")
if os.path.exists(err_path):
    err_count = len(pd.read_csv(err_path)) if os.path.getsize(err_path) > 0 else 0
    print(f"[errors] {err_count} errors logged -> {err_path}")

In [ ]:
# Step 5: Compute evaluation metrics for FSRL
%pip install -q scikit-learn

import pandas as pd
import json
import time
from sklearn.metrics import accuracy_score, classification_report
from collections import Counter

# 加载预测结果
df = pd.read_csv(pred_path).sort_values("row_idx").drop_duplicates("row_idx", keep="last")

# 解析标签
def parse_labels(s):
    if pd.isna(s) or s == '[]' or s == '':
        return []
    try:
        return json.loads(s)
    except:
        return []

df['gold_parsed'] = df['gold_labels'].apply(parse_labels)
df['pred_parsed'] = df['pred_labels'].apply(parse_labels)

# 只考虑成功的预测
success_df = df[df['success'] == True].copy() if 'success' in df.columns else df.copy()
print(f"Total samples: {len(df)}")
print(f"Successful predictions: {len(success_df)}")
print(f"Failed predictions: {len(df) - len(success_df)}")

if len(success_df) > 0:
    # 收集所有token级别的标签
    all_true = []
    all_pred = []
    
    for _, row in success_df.iterrows():
        gold = row['gold_parsed']
        pred = row['pred_parsed']
        
        min_len = min(len(gold), len(pred))
        all_true.extend(gold[:min_len])
        all_pred.extend(pred[:min_len])
    
    # 定义所有可能的标签
    labels = ["O", "B-QUANT", "I-QUANT", "B-TIME", "I-TIME", "B-MANNER", "I-MANNER",
              "B-THEME", "I-THEME", "B-VALUE", "I-VALUE", "B-WHOLE", "I-WHOLE",
              "B-LOCATION", "I-LOCATION", "B-AGENT", "I-AGENT", "B-CAUSE", "I-CAUSE",
              "B-SOURCE", "I-SOURCE", "B-REF_TIME", "I-REF_TIME", "B-CONDITION", "I-CONDITION"]
    
    # 计算指标
    accuracy = accuracy_score(all_true, all_pred)
    
    print("\n" + "="*50)
    print("EVALUATION RESULTS - o3-mini on FLARE-FSRL")
    print("="*50)
    print(f"Total Tokens: {len(all_true)}")
    print(f"Accuracy: {accuracy:.4f}")
    
    # 标签分布
    label_counts = Counter(all_true)
    print(f"\n真实标签分布 (Top 15):")
    for label, count in label_counts.most_common(15):
        print(f"  {label}: {count} ({count/len(all_true)*100:.2f}%)")
    
    # 二值化评估（O vs 非O）
    true_binary = ["非O" if l != "O" else "O" for l in all_true]
    pred_binary = ["非O" if l != "O" else "O" for l in all_pred]
    binary_accuracy = accuracy_score(true_binary, pred_binary)
    print(f"\n二值化准确率 (O vs 非O): {binary_accuracy:.4f}")
    
    # 非O标签统计
    true_non_o = sum(1 for l in all_true if l != "O")
    pred_non_o = sum(1 for l in all_pred if l != "O")
    correct_non_o = sum(1 for t, p in zip(all_true, all_pred) if t != "O" and t == p)
    print(f"\n非O标签统计:")
    print(f"  真实非O数量: {true_non_o}")
    print(f"  预测非O数量: {pred_non_o}")
    print(f"  精确匹配的非O: {correct_non_o}")
    
    # 分类报告
    print("\nClassification Report:")
    print(classification_report(all_true, all_pred, labels=labels, zero_division=0))
    
    # 保存评估结果
    eval_results = {
        "model": MODEL,
        "dataset": "ChanceFocus/flare-fsrl",
        "split": "test",
        "total_samples": len(df),
        "successful_predictions": len(success_df),
        "failed_predictions": len(df) - len(success_df),
        "total_tokens": len(all_true),
        "accuracy": float(accuracy),
        "binary_accuracy": float(binary_accuracy),
        "true_non_o": true_non_o,
        "pred_non_o": pred_non_o,
        "correct_non_o": correct_non_o,
        "label_distribution": {k: v for k, v in label_counts.most_common(30)},
        "classification_report": classification_report(all_true, all_pred, labels=labels, zero_division=0, output_dict=True),
        "evaluation_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())
    }
    
    eval_path = f"{save_dir}/{run_tag}_evaluation_results.json"
    with open(eval_path, "w") as f:
        json.dump(eval_results, f, indent=2)
    print(f"\nEvaluation results saved -> {eval_path}")
    
else:
    print("No successful predictions to evaluate!")